# Ten PPO runs, one fake win, zero real ones

I ran PPO on this game across ten configurations. None of them beat the frozen policy they started
from. That result is not interesting by itself — plenty of PPO runs fail. What is worth writing
down is the four specific ways it failed, because three of them are invisible from the training
curves and one of them briefly produced a *positive* result that was not real.

If you are about to point RL at this environment, these are the four traps, with the numbers.

## Trap 1: the reward has no gradient because the economy collapses first

The first runs learned nothing at all, and the loss curves looked fine. The cause was upstream of
the optimiser: from a random initialisation the policy breaks the financial chain in the first few
days, every rollout ends in the same bankrupt state, and every trajectory therefore receives the
same terminal reward. There is no signal to ascend. The agent was not learning slowly — the
objective was flat.

Two things fixed it. Initialise from a policy that already runs the economy, and make the terminal
reward *graded* rather than win/lose, so that being less bankrupt is measurably better than being
more bankrupt.

The general form: in a long economic episode, check that your reward actually varies across
rollouts before you tune anything. If the spread is zero, no amount of entropy or learning rate
will help.

## Trap 2: the entropy bonus quietly ate the anchor

With an anchored initialisation the runs finally moved — in the wrong direction. The entropy bonus
does what it says, and what it says is "be less like the thing you started from".

I logged the probability mass the policy put on the neutral action (the one that reproduces the
anchor's behaviour):

| | mass on the neutral action |
|---|---|
| at initialisation | **91.6%** |
| after training with an entropy bonus | **46.5%** |

Half the anchor was gone, and with it the economy. Replacing the entropy bonus with an adaptive KL
leash back to the anchor kept exploration bounded to the region where the farm still functions.

## Trap 3: the critic warm-up moved the policy it was supposed to be measuring

Standard advice: freeze the policy and train the value head first, so advantages are not noise.
I did that and the runs got worse.

The value head shared the trunk with the policy head. "Freezing the policy" froze the policy
*parameters*, but training the critic still updated the shared trunk, so the policy's actual output
drifted — to a KL of **3.258** from where it started, during a phase whose entire purpose was to
change nothing.

One character fixes it: read the value head off `trunk.detach()`. Worth checking in any shared-trunk
actor-critic, not just this game.

## Trap 4: the fake win

This is the one I would most like to have caught earlier.

A run reported **+8,750 coins, t = 2.94** against baseline. It survived a re-run. I nearly built on
it.

The evaluation harness scored the baseline by forcing the policy's action to index 2 of the action
grid, on the assumption that index 2 was the neutral multiplier 1.0. That had been true of the
coarse grid. I had since replaced it with a finer one:

```
coarse grid: [0, 0.5, 1.0, 1.5, 2.0, 3.0]    -> index 2 is 1.0   (neutral)
fine grid:   [0.8, 0.9, 0.95, 1.0, 1.05, ...] -> index 2 is 0.95  (NOT neutral)
```

So the "baseline" was selling everything 5% below the neutral price, all season. The candidate was
not winning; the baseline was being handicapped. The measured effect was real, repeatable, and
entirely an artefact.

The fix in the harness is one line, and it is the shape of fix worth copying:

In [ ]:
# Before: a constant that was true when it was written
NEUTRAL = 2

# After: derived from the grid, so it cannot drift out of sync when the grid changes
import numpy as np

LEVELS = np.array([0.80, 0.90, 0.95, 1.00, 1.05, 1.10, 1.20, 1.40])
NEUTRAL = int((LEVELS == 1.0).argmax())
assert LEVELS[NEUTRAL] == 1.0, "no neutral action in the grid"

print(f"neutral index = {NEUTRAL}, multiplier = {LEVELS[NEUTRAL]}")

# The general check: any index into a table that encodes a MEANING should be derived from the
# table, and asserted. A hard-coded index is correct until someone edits the table, and then it
# is silently wrong in a direction that flatters whichever side it handicaps.

## What I think the real obstacle is

After the four fixes, ten configurations, none beating the anchor, I stopped and measured something
else instead: whether the policy is separable at all. It is not. Playing one policy's farm actions
with another's market orders earns **0 coins** — the pair is worth ~108,000 together and nothing
apart. I wrote that up
[here](https://www.kaggle.com/code/zhincez/the-market-half-is-worth-zero-on-its-own).

That reframes the RL failure. The thing to be learned is a 720-turn plan whose two halves stay
consistent, and per-step credit assignment through a 720-step economic chain is a hard way to
discover that consistency when the reward only arrives at the end.

I am not claiming RL cannot work here. I am claiming that if you try it, the four traps above will
cost you days each, and the fourth one will cost you a result you believe. If you get PPO to beat
its own anchor on this environment, please post it — I would like to know which of my four
conclusions was the wrong one.